# Titanic Dataset Analysis
### AI Assignment 2 — Data Cleaning · Feature Engineering · Feature Selection

**Dataset:** [Kaggle Titanic — Machine Learning from Disaster](https://www.kaggle.com/c/titanic)

---


## 0. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)

df_raw = pd.read_csv('../data/train.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()


---
## Part 1 — Data Cleaning

### 1.1 Missing Value Analysis


In [ ]:
missing = df_raw.isnull().sum()
pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': pct}).query('`Missing Count` > 0')


### Decision rationale

| Column | Missing % | Strategy | Reason |
|--------|-----------|----------|--------|
| `Age` | ~20% | Median imputation + `HasAge` flag | Median is robust against outliers; the binary flag lets the model learn that missingness itself may be informative |
| `Embarked` | <1% | Mode imputation | Only 2 rows missing; 'S' (Southampton) is overwhelmingly the mode |
| `Fare` | <1% | Median imputation | One missing in test set; median prevents outlier skew |
| `Cabin` | ~77% | Keep raw for deck extraction | Too sparse to impute; we extract the deck letter as a categorical signal |


In [ ]:
from data_cleaning import clean, report_missing

df = df_raw.copy()

# Add HasAge flag before filling
df['HasAge'] = df['Age'].notna().astype(int)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

print('Remaining missing values:')
print(report_missing(df))


### 1.2 Outlier Detection & Handling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df_raw['Fare'].dropna(), ax=axes[0])
axes[0].set_title('Fare — Before Capping')

sns.boxplot(x=df_raw['Age'].dropna(), ax=axes[1])
axes[1].set_title('Age — Before Capping')

plt.tight_layout()
plt.show()

print(f"Fare 99th percentile: {df_raw['Fare'].quantile(0.99):.2f}")
print(f"Fare max: {df_raw['Fare'].max():.2f}")


**Strategy:** Cap `Fare` at the 99th percentile. One passenger paid £512 — likely a data entry quirk or a shared luxury suite split differently. We cap rather than drop so we keep the survival label. `Age` is biologically bounded; values outside [0, 80] are capped.


In [ ]:
fare_cap = df['Fare'].quantile(0.99)
df['Fare'] = df['Fare'].clip(upper=fare_cap)
df['Age'] = df['Age'].clip(lower=0, upper=80)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df['Fare'], ax=axes[0]); axes[0].set_title('Fare — After Capping')
sns.boxplot(x=df['Age'], ax=axes[1]); axes[1].set_title('Age — After Capping')
plt.tight_layout(); plt.show()


### 1.3 Data Consistency & Duplicates

In [ ]:
print('Sex unique values:', df_raw['Sex'].unique())
print('Embarked unique values:', df_raw['Embarked'].unique())

# Normalise strings
df['Sex'] = df['Sex'].str.strip().str.lower()
df['Embarked'] = df['Embarked'].str.strip().str.upper()

dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')
if dupes:
    df = df.drop_duplicates()


### 1.4 Save Cleaned Dataset

In [ ]:
df.to_csv('../data/train_cleaned.csv', index=False)
print(f'Saved train_cleaned.csv — shape: {df.shape}')
df.dtypes


---
## Part 2 — Feature Engineering

### 2.1 Family Features


In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=df, x='FamilySize', y='Survived', ax=axes[0], errorbar=None)
axes[0].set_title('Survival Rate by Family Size')

sns.barplot(data=df, x='IsAlone', y='Survived', ax=axes[1], errorbar=None)
axes[1].set_title('Survival Rate: Solo vs Group')

plt.tight_layout(); plt.show()


**Insight:** Solo travellers survive at a notably lower rate. Very large families (6+) also struggle — likely couldn't co-ordinate evacuation. The sweet spot is families of 2–4.


### 2.2 Title Extraction

In [ ]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
print(df['Title'].value_counts())


In [ ]:
rare = {'Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'}
df['Title'] = df['Title'].replace(list(rare), 'Rare')
df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

fig, ax = plt.subplots(figsize=(10, 4))
survival = df.groupby('Title')['Survived'].mean().sort_values()
survival.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Survival Rate by Title')
ax.set_xlabel('Survival Rate')
plt.tight_layout(); plt.show()


**Insight:** `Mrs` and `Miss` survive at much higher rates — consistent with the 'women and children first' evacuation policy. `Master` (young boys) also has a relatively high rate. Title captures gender + social status more precisely than `Sex` alone.


### 2.3 Deck Extraction from Cabin

In [ ]:
df['Deck'] = df['Cabin'].str[0].fillna('Unknown')
print(df['Deck'].value_counts())

fig, ax = plt.subplots(figsize=(10, 4))
df.groupby('Deck')['Survived'].mean().sort_values().plot(kind='barh', ax=ax, color='coral')
ax.set_title('Survival Rate by Deck')
plt.tight_layout(); plt.show()


**Note:** ~77% of cabin values are missing. Passengers without a cabin are grouped as `Unknown`. Despite the sparsity, the deck letter correlates with class and proximity to lifeboats.


### 2.4 Age Groups

In [ ]:
bins   = [0, 12, 17, 60, 80]
labels = ['Child', 'Teen', 'Adult', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)

fig, ax = plt.subplots(figsize=(8, 4))
df.groupby('AgeGroup', observed=True)['Survived'].mean().plot(kind='bar', ax=ax, rot=0, color='mediumseagreen')
ax.set_title('Survival Rate by Age Group')
ax.set_ylabel('Survival Rate')
plt.tight_layout(); plt.show()


### 2.5 Fare Per Person

In [ ]:
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
survivors   = df[df['Survived'] == 1]['FarePerPerson']
non_survivors = df[df['Survived'] == 0]['FarePerPerson']

axes[0].hist(survivors, bins=30, alpha=0.6, label='Survived', color='steelblue')
axes[0].hist(non_survivors, bins=30, alpha=0.6, label='Died', color='coral')
axes[0].set_title('FarePerPerson Distribution')
axes[0].legend()

sns.boxplot(data=df, x='Survived', y='FarePerPerson', ax=axes[1])
axes[1].set_title('FarePerPerson by Survival')

plt.tight_layout(); plt.show()


### 2.6 Log Transformation of Skewed Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df['Fare'].hist(bins=50, ax=axes[0,0]); axes[0,0].set_title('Fare — Original')
np.log1p(df['Fare']).hist(bins=50, ax=axes[0,1]); axes[0,1].set_title('Fare — log1p')
df['FarePerPerson'].hist(bins=50, ax=axes[1,0]); axes[1,0].set_title('FarePerPerson — Original')
np.log1p(df['FarePerPerson']).hist(bins=50, ax=axes[1,1]); axes[1,1].set_title('FarePerPerson — log1p')

plt.tight_layout(); plt.show()

df['Fare_log']          = np.log1p(df['Fare'])
df['FarePerPerson_log'] = np.log1p(df['FarePerPerson'])


**Rationale:** Fare is heavily right-skewed (Gini coefficient > 0.5 for this dataset). Log transformation compresses the long tail and makes the feature more useful for linear and distance-based models. Tree models are less affected but it doesn't hurt.


### 2.7 Interaction Features (Optional)

In [ ]:
df['Pclass_x_Fare'] = df['Pclass'] * df['Fare']

fig, ax = plt.subplots(figsize=(8, 4))
df.groupby('Pclass')['Fare'].mean().plot(kind='bar', ax=ax, rot=0)
ax.set_title('Mean Fare by Pclass')
plt.tight_layout(); plt.show()


### 2.8 Categorical Encoding

In [ ]:
nominal_cols = ['Sex', 'Embarked', 'Title', 'Deck', 'AgeGroup']
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(f'Columns before encoding: {df.shape[1]}')
print(f'Columns after encoding:  {df_encoded.shape[1]}')
df_encoded.head(3)


---
## Part 3 — Feature Selection

### 3.1 Correlation Analysis


In [ ]:
# Keep only numeric columns with the target
drop_id = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_model = df_encoded.drop(columns=[c for c in drop_id if c in df_encoded.columns], errors='ignore')

# Encode any residual category dtype
for col in df_model.select_dtypes(include='category').columns:
    df_model[col] = df_model[col].cat.codes

df_model = df_model.fillna(0)

fig, ax = plt.subplots(figsize=(16, 14))
corr_matrix = df_model.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout(); plt.show()


In [ ]:
# Find pairs above 0.90 threshold
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, row, round(upper.loc[row, col], 3))
             for col in upper.columns
             for row in upper.index
             if upper.loc[row, col] > 0.9]

print('Highly correlated pairs (|r| > 0.90):')
for a, b, r in high_corr:
    print(f'  {a} ↔ {b}  r={r}')


### 3.2 Random Forest Feature Importance

In [ ]:
X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importances.head(20).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Feature Importances (Random Forest)')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout(); plt.show()

print('Top 20 features:')
print(importances.head(20).to_string())


### 3.3 Recursive Feature Elimination (RFE)

In [ ]:
rfe = RFE(estimator=RandomForestClassifier(n_estimators=100, random_state=42),
          n_features_to_select=15, step=1)
rfe.fit(X, y)
rfe_selected = X.columns[rfe.support_].tolist()

print(f'RFE selected {len(rfe_selected)} features:')
for f in sorted(rfe_selected):
    print(f'  - {f}')


### 3.4 Final Feature Set

Combining RF importance (importance > 0.01) and RFE consensus:


In [ ]:
importance_threshold = 0.01
rf_selected = importances[importances > importance_threshold].index.tolist()

# Intersection of RF and RFE for high confidence
consensus = list(set(rf_selected) & set(rfe_selected))
print(f'RF selected ({len(rf_selected)}):  {sorted(rf_selected)}')
print()
print(f'RFE selected ({len(rfe_selected)}): {sorted(rfe_selected)}')
print()
print(f'Consensus ({len(consensus)}): {sorted(consensus)}')


### Feature Selection Summary

| Feature | Kept? | Reason |
|---------|-------|--------|
| `Title_*` | ✅ | High RF importance; captures gender + social class compactly |
| `Fare_log` / `FarePerPerson_log` | ✅ | Strong signal; log form preferred over raw |
| `Pclass` | ✅ | One of the top predictors |
| `Age` | ✅ | Continuous signal; child penalty well established |
| `Sex_male` | ✅ | Strongest single predictor |
| `FamilySize` / `IsAlone` | ✅ | Both selected; mild redundancy acceptable given importance |
| `Deck_*` | ⚠️ Partial | Keep decks A–C; Unknown adds noise |
| `SibSp`, `Parch` | ❌ | Subsumed by `FamilySize`; drop to reduce multicollinearity |
| `Pclass_x_Fare` | ❌ | Correlated with both parent features; marginal gain |
| `HasAge` | ⚠️ | Low importance; keep only if missing data pattern matters |


In [ ]:
# Final model-ready DataFrame
final_features = consensus if consensus else rf_selected  # fallback
X_final = X[final_features]
print(f'Final dataset shape: {X_final.shape}')
X_final.head(3)


---
## Summary

This notebook covered:
1. **Data Cleaning** — median imputation for Age/Fare, mode for Embarked, 99th-percentile capping for Fare outliers
2. **Feature Engineering** — FamilySize, IsAlone, Title, Deck, AgeGroup, FarePerPerson, log transforms, and one interaction term
3. **Feature Selection** — correlation filter (r > 0.90), RF importance ranking, and RFE for final feature list

Key takeaway: `Sex`, `Title`, `Pclass`, `Fare_log`, and `FamilySize` dominate survival prediction, consistent with the historical record.
